# LSTM-based Intrusion Detection System Training

This notebook demonstrates the training and evaluation of LSTM models for real-time intrusion detection using CIC-IDS features.

## Table of Contents
1. [Data Loading and Preprocessing](#data-loading)
2. [Feature Engineering](#feature-engineering)
3. [Model Architecture](#model-architecture)
4. [Training Process](#training)
5. [Model Evaluation](#evaluation)
6. [Real-time Testing](#real-time)
7. [Model Deployment](#deployment)

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 1. Data Loading and Preprocessing {#data-loading}

In [ ]:
# Load CIC-IDS dataset
def load_cicids_data(file_path='../sample_security_data.csv'):
    """
    Load and preprocess CIC-IDS dataset
    """
    try:
        df = pd.read_csv(file_path)
        print(f"Loaded dataset with shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        return df
    except FileNotFoundError:
        print("Dataset not found. Generating synthetic data...")
        return generate_synthetic_data()

def generate_synthetic_data(n_samples=10000):
    """
    Generate synthetic network flow data for demonstration
    """
    np.random.seed(42)
    
    # Generate normal traffic patterns
    normal_data = {
        'flow_duration': np.random.exponential(1000, n_samples // 2),
        'total_fwd_packets': np.random.poisson(50, n_samples // 2),
        'total_bwd_packets': np.random.poisson(45, n_samples // 2),
        'total_length_fwd_packets': np.random.normal(1500, 500, n_samples // 2),
        'total_length_bwd_packets': np.random.normal(1400, 400, n_samples // 2),
        'flow_bytes_s': np.random.normal(10000, 3000, n_samples // 2),
        'flow_packets_s': np.random.normal(100, 30, n_samples // 2),
        'flow_iat_mean': np.random.normal(1000, 300, n_samples // 2),
        'flow_iat_std': np.random.normal(500, 200, n_samples // 2),
        'fwd_header_length': np.random.normal(20, 5, n_samples // 2),
        'psh_flag_count': np.random.poisson(5, n_samples // 2),
        'urg_flag_count': np.random.poisson(0.1, n_samples // 2),
        'packet_length_mean': np.random.normal(800, 200, n_samples // 2),
        'packet_length_std': np.random.normal(300, 100, n_samples // 2),
        'down_up_ratio': np.random.normal(1.0, 0.3, n_samples // 2),
        'average_packet_size': np.random.normal(750, 150, n_samples // 2),
        'avg_fwd_segment_size': np.random.normal(800, 200, n_samples // 2),
        'subflow_fwd_packets': np.random.poisson(25, n_samples // 2),
        'active_mean': np.random.normal(5000, 1000, n_samples // 2),
        'idle_mean': np.random.normal(2000, 500, n_samples // 2),
        'label': np.zeros(n_samples // 2)
    }
    
    # Generate attack traffic patterns (anomalous)
    attack_data = {
        'flow_duration': np.random.exponential(5000, n_samples // 2),  # Longer duration
        'total_fwd_packets': np.random.poisson(200, n_samples // 2),  # More packets
        'total_bwd_packets': np.random.poisson(5, n_samples // 2),    # Fewer response packets
        'total_length_fwd_packets': np.random.normal(5000, 1000, n_samples // 2),
        'total_length_bwd_packets': np.random.normal(500, 200, n_samples // 2),
        'flow_bytes_s': np.random.normal(50000, 10000, n_samples // 2),  # High bandwidth
        'flow_packets_s': np.random.normal(500, 100, n_samples // 2),   # High packet rate
        'flow_iat_mean': np.random.normal(100, 50, n_samples // 2),     # Low IAT
        'flow_iat_std': np.random.normal(50, 20, n_samples // 2),
        'fwd_header_length': np.random.normal(25, 5, n_samples // 2),
        'psh_flag_count': np.random.poisson(20, n_samples // 2),        # More PSH flags
        'urg_flag_count': np.random.poisson(2, n_samples // 2),         # More URG flags
        'packet_length_mean': np.random.normal(1200, 300, n_samples // 2),
        'packet_length_std': np.random.normal(600, 200, n_samples // 2),
        'down_up_ratio': np.random.normal(10.0, 3.0, n_samples // 2),  # High ratio
        'average_packet_size': np.random.normal(1100, 200, n_samples // 2),
        'avg_fwd_segment_size': np.random.normal(1200, 300, n_samples // 2),
        'subflow_fwd_packets': np.random.poisson(100, n_samples // 2),
        'active_mean': np.random.normal(15000, 3000, n_samples // 2),
        'idle_mean': np.random.normal(500, 200, n_samples // 2),
        'label': np.ones(n_samples // 2)
    }
    
    # Combine normal and attack data
    data = {}
    for key in normal_data.keys():
        data[key] = np.concatenate([normal_data[key], attack_data[key]])
    
    df = pd.DataFrame(data)
    
    # Shuffle the dataset
    df = df.sample(frac=1).reset_index(drop=True)
    
    print(f"Generated synthetic dataset with shape: {df.shape}")
    print(f"Class distribution: {df['label'].value_counts()}")
    
    return df

# Load data
df = load_cicids_data()
df.head()

In [ ]:
# Data exploration and visualization
print("Dataset Information:")
print(f"Shape: {df.shape}")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\nClass distribution:")
print(df['label'].value_counts())

# Visualize class distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
df['label'].value_counts().plot(kind='bar')
plt.title('Class Distribution')
plt.xlabel('Class (0=Normal, 1=Attack)')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
plt.pie(df['label'].value_counts(), labels=['Normal', 'Attack'], autopct='%1.1f%%')
plt.title('Class Distribution (Percentage)')

plt.tight_layout()
plt.show()

## 2. Feature Engineering {#feature-engineering}

In [ ]:
# Feature correlation analysis
def analyze_features(df):
    """
    Analyze feature correlations and importance
    """
    # Select numeric features
    numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'label' in numeric_features:
        numeric_features.remove('label')
    
    # Correlation matrix
    plt.figure(figsize=(15, 12))
    correlation_matrix = df[numeric_features].corr()
    sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0)
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()
    
    # Feature distributions by class
    fig, axes = plt.subplots(4, 5, figsize=(20, 16))
    axes = axes.ravel()
    
    for i, feature in enumerate(numeric_features[:20]):
        df[df['label'] == 0][feature].hist(alpha=0.5, label='Normal', bins=30, ax=axes[i])
        df[df['label'] == 1][feature].hist(alpha=0.5, label='Attack', bins=30, ax=axes[i])
        axes[i].set_title(feature)
        axes[i].legend()
    
    plt.tight_layout()
    plt.show()
    
    return numeric_features

feature_columns = analyze_features(df)

In [ ]:
# Prepare features and labels
def prepare_data(df, feature_columns, sequence_length=10):
    """
    Prepare data for LSTM training
    """
    # Extract features and labels
    X = df[feature_columns].values
    y = df['label'].values
    
    # Handle missing values
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Normalize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Create sequences for LSTM
    X_sequences = []
    y_sequences = []
    
    for i in range(sequence_length, len(X_scaled)):
        X_sequences.append(X_scaled[i-sequence_length:i])
        y_sequences.append(y[i])
    
    X_sequences = np.array(X_sequences)
    y_sequences = np.array(y_sequences)
    
    print(f"Sequence shape: {X_sequences.shape}")
    print(f"Labels shape: {y_sequences.shape}")
    
    return X_sequences, y_sequences, scaler

# Prepare sequences
sequence_length = 10
X_sequences, y_sequences, scaler = prepare_data(df, feature_columns, sequence_length)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_sequences, y_sequences, test_size=0.2, random_state=42, stratify=y_sequences
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"Training set: {X_train.shape}, {y_train.shape}")
print(f"Validation set: {X_val.shape}, {y_val.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")

## 3. Model Architecture {#model-architecture}

In [ ]:
def create_lstm_model(input_shape, architecture='deep'):
    """
    Create LSTM model for intrusion detection
    """
    model = keras.Sequential()
    
    if architecture == 'simple':
        model.add(layers.LSTM(32, input_shape=input_shape))
        model.add(layers.Dropout(0.2))
        model.add(layers.Dense(16, activation='relu'))
        model.add(layers.Dense(1, activation='sigmoid'))
        
    elif architecture == 'deep':
        model.add(layers.LSTM(64, return_sequences=True, input_shape=input_shape))
        model.add(layers.Dropout(0.3))
        model.add(layers.LSTM(32, return_sequences=True))
        model.add(layers.Dropout(0.3))
        model.add(layers.LSTM(16))
        model.add(layers.Dropout(0.2))
        model.add(layers.Dense(8, activation='relu'))
        model.add(layers.Dense(1, activation='sigmoid'))
        
    elif architecture == 'bidirectional':
        model.add(layers.Bidirectional(layers.LSTM(32, return_sequences=True), input_shape=input_shape))
        model.add(layers.Dropout(0.3))
        model.add(layers.Bidirectional(layers.LSTM(16)))
        model.add(layers.Dropout(0.2))
        model.add(layers.Dense(8, activation='relu'))
        model.add(layers.Dense(1, activation='sigmoid'))
    
    return model

# Create models with different architectures
input_shape = (sequence_length, len(feature_columns))
print(f"Input shape: {input_shape}")

models = {
    'simple': create_lstm_model(input_shape, 'simple'),
    'deep': create_lstm_model(input_shape, 'deep'),
    'bidirectional': create_lstm_model(input_shape, 'bidirectional')
}

# Display model architecture
for name, model in models.items():
    print(f"\n{name.upper()} MODEL:")
    model.summary()
    print("-" * 50)

## 4. Training Process {#training}

In [ ]:
def compile_and_train_model(model, X_train, y_train, X_val, y_val, model_name):
    """
    Compile and train LSTM model
    """
    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )
    
    # Callbacks
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=0.0001
        ),
        keras.callbacks.ModelCheckpoint(
            f'../models/lstm_ids_{model_name}.h5',
            monitor='val_loss',
            save_best_only=True
        )
    ]
    
    # Train model
    history = model.fit(
        X_train, y_train,
        batch_size=32,
        epochs=50,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        verbose=1
    )
    
    return history

# Train all models
histories = {}

for name, model in models.items():
    print(f"\nTraining {name.upper()} model...")
    history = compile_and_train_model(model, X_train, y_train, X_val, y_val, name)
    histories[name] = history
    print(f"Training completed for {name} model")

In [ ]:
# Plot training history
def plot_training_history(histories):
    """
    Plot training history for all models
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    metrics = ['loss', 'accuracy', 'precision', 'recall']
    
    for i, metric in enumerate(metrics):
        ax = axes[i//2, i%2]
        
        for name, history in histories.items():
            ax.plot(history.history[metric], label=f'{name} train')
            ax.plot(history.history[f'val_{metric}'], label=f'{name} val', linestyle='--')
        
        ax.set_title(f'{metric.capitalize()} History')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(metric.capitalize())
        ax.legend()
        ax.grid(True)
    
    plt.tight_layout()
    plt.show()

plot_training_history(histories)

## 5. Model Evaluation {#evaluation}

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """
    Comprehensive model evaluation
    """
    # Predictions
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()
    y_pred_prob = y_pred_prob.flatten()
    
    # Classification report
    print(f"\n{model_name.upper()} MODEL EVALUATION:")
    print("=" * 50)
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))
    
    # ROC AUC
    auc_score = roc_auc_score(y_test, y_pred_prob)
    print(f"ROC AUC Score: {auc_score:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(12, 4))
    
    # Confusion Matrix
    plt.subplot(1, 2, 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Normal', 'Attack'], 
                yticklabels=['Normal', 'Attack'])
    plt.title(f'{model_name} - Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    # ROC Curve
    plt.subplot(1, 2, 2)
    fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {auc_score:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} - ROC Curve')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    return {
        'predictions': y_pred,
        'probabilities': y_pred_prob,
        'auc_score': auc_score,
        'confusion_matrix': cm
    }

# Evaluate all models
evaluation_results = {}

for name, model in models.items():
    results = evaluate_model(model, X_test, y_test, name)
    evaluation_results[name] = results

In [ ]:
# Compare model performance
def compare_models(evaluation_results):
    """
    Compare performance of all models
    """
    comparison_data = []
    
    for name, results in evaluation_results.items():
        cm = results['confusion_matrix']
        tn, fp, fn, tp = cm.ravel()
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        
        comparison_data.append({
            'Model': name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1_score,
            'AUC': results['auc_score']
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\nMODEL COMPARISON:")
    print("=" * 60)
    print(comparison_df.round(4))
    
    # Visualization
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']
    
    plt.figure(figsize=(12, 8))
    
    x = np.arange(len(comparison_df))
    width = 0.15
    
    for i, metric in enumerate(metrics):
        plt.bar(x + i * width, comparison_df[metric], width, label=metric)
    
    plt.xlabel('Models')
    plt.ylabel('Score')
    plt.title('Model Performance Comparison')
    plt.xticks(x + width * 2, comparison_df['Model'])
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return comparison_df

comparison_results = compare_models(evaluation_results)

## 6. Real-time Testing {#real-time}

In [ ]:
# Select best model based on AUC score
best_model_name = comparison_results.loc[comparison_results['AUC'].idxmax(), 'Model']
best_model = models[best_model_name]

print(f"Best performing model: {best_model_name}")
print(f"AUC Score: {comparison_results.loc[comparison_results['AUC'].idxmax(), 'AUC']:.4f}")

# Real-time prediction simulation
def simulate_real_time_detection(model, scaler, sequence_length=10, n_samples=100):
    """
    Simulate real-time attack detection
    """
    print("\nSimulating real-time detection...")
    
    # Generate streaming data
    streaming_data = generate_synthetic_data(n_samples)
    features = streaming_data[feature_columns].values
    features_scaled = scaler.transform(np.nan_to_num(features))
    true_labels = streaming_data['label'].values
    
    # Real-time detection simulation
    detection_results = []
    
    # Initialize with first sequence
    current_sequence = deque(maxlen=sequence_length)
    for i in range(sequence_length):
        current_sequence.append(features_scaled[i])
    
    # Process remaining samples
    for i in range(sequence_length, len(features_scaled)):
        # Prepare sequence for prediction
        sequence_array = np.array(list(current_sequence)).reshape(1, sequence_length, -1)
        
        # Make prediction
        prediction_prob = model.predict(sequence_array, verbose=0)[0][0]
        prediction = 1 if prediction_prob > 0.5 else 0
        
        # Store results
        detection_results.append({
            'sample_id': i,
            'true_label': true_labels[i],
            'predicted_label': prediction,
            'prediction_prob': prediction_prob,
            'is_attack_detected': prediction == 1 and true_labels[i] == 1,
            'is_false_positive': prediction == 1 and true_labels[i] == 0
        })
        
        # Update sequence
        current_sequence.append(features_scaled[i])
    
    # Analysis
    results_df = pd.DataFrame(detection_results)
    
    attacks_detected = results_df['is_attack_detected'].sum()
    false_positives = results_df['is_false_positive'].sum()
    total_attacks = (results_df['true_label'] == 1).sum()
    total_normal = (results_df['true_label'] == 0).sum()
    
    print(f"\nReal-time Detection Results:")
    print(f"Total samples processed: {len(detection_results)}")
    print(f"Total attacks in stream: {total_attacks}")
    print(f"Attacks detected: {attacks_detected}")
    print(f"Detection rate: {attacks_detected/total_attacks*100:.2f}%")
    print(f"False positives: {false_positives}")
    print(f"False positive rate: {false_positives/total_normal*100:.2f}%")
    
    # Visualization
    plt.figure(figsize=(15, 10))
    
    # Real-time detection plot
    plt.subplot(2, 2, 1)
    plt.plot(results_df['sample_id'], results_df['prediction_prob'], alpha=0.7, label='Prediction Probability')
    plt.axhline(y=0.5, color='r', linestyle='--', label='Threshold')
    plt.scatter(results_df[results_df['true_label']==1]['sample_id'], 
               results_df[results_df['true_label']==1]['prediction_prob'], 
               color='red', alpha=0.5, label='True Attacks')
    plt.xlabel('Sample ID')
    plt.ylabel('Prediction Probability')
    plt.title('Real-time Attack Detection')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Detection timeline
    plt.subplot(2, 2, 2)
    attack_times = results_df[results_df['is_attack_detected']]['sample_id']
    fp_times = results_df[results_df['is_false_positive']]['sample_id']
    
    plt.scatter(attack_times, [1]*len(attack_times), color='red', label='Attacks Detected', s=50)
    plt.scatter(fp_times, [0]*len(fp_times), color='orange', label='False Positives', s=50)
    plt.xlabel('Sample ID')
    plt.ylabel('Detection Type')
    plt.title('Detection Timeline')
    plt.yticks([0, 1], ['False Positive', 'Attack Detected'])
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Prediction distribution
    plt.subplot(2, 2, 3)
    normal_probs = results_df[results_df['true_label']==0]['prediction_prob']
    attack_probs = results_df[results_df['true_label']==1]['prediction_prob']
    
    plt.hist(normal_probs, bins=20, alpha=0.5, label='Normal Traffic', color='blue')
    plt.hist(attack_probs, bins=20, alpha=0.5, label='Attack Traffic', color='red')
    plt.axvline(x=0.5, color='black', linestyle='--', label='Threshold')
    plt.xlabel('Prediction Probability')
    plt.ylabel('Frequency')
    plt.title('Prediction Distribution')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Performance metrics over time
    plt.subplot(2, 2, 4)
    window_size = 20
    rolling_accuracy = []
    
    for i in range(window_size, len(results_df)):
        window = results_df.iloc[i-window_size:i]
        accuracy = (window['true_label'] == window['predicted_label']).mean()
        rolling_accuracy.append(accuracy)
    
    plt.plot(range(window_size, len(results_df)), rolling_accuracy)
    plt.xlabel('Sample ID')
    plt.ylabel('Rolling Accuracy')
    plt.title(f'Rolling Accuracy (Window Size: {window_size})')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return results_df

# Run real-time simulation
real_time_results = simulate_real_time_detection(best_model, scaler)

## 7. Model Deployment {#deployment}

In [ ]:
# Save the best model and preprocessing components
import os
import joblib

# Create models directory
os.makedirs('../models', exist_ok=True)

# Save best model
best_model.save(f'../models/best_lstm_ids_{best_model_name}.h5')
print(f"Saved best model: ../models/best_lstm_ids_{best_model_name}.h5")

# Save scaler
joblib.dump(scaler, '../models/feature_scaler.pkl')
print("Saved feature scaler: ../models/feature_scaler.pkl")

# Save feature names
with open('../models/feature_names.txt', 'w') as f:
    for feature in feature_columns:
        f.write(f"{feature}\n")
print("Saved feature names: ../models/feature_names.txt")

# Save model configuration
model_config = {
    'best_model': best_model_name,
    'sequence_length': sequence_length,
    'num_features': len(feature_columns),
    'threshold': 0.5,
    'performance': comparison_results.to_dict('records')
}

import json
with open('../models/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print("Saved model configuration: ../models/model_config.json")

print("\nModel deployment files created successfully!")
print("\nFiles created:")
for file in os.listdir('../models'):
    print(f"  - ../models/{file}")

In [ ]:
# Create deployment script
deployment_script = '''
#!/usr/bin/env python3
"""
LSTM IDS Model Loader for Real-time Detection
Usage: from lstm_model_loader import LSTMIDSModel
"""

import numpy as np
import tensorflow as tf
from tensorflow import keras
import joblib
import json
from collections import deque

class LSTMIDSModel:
    def __init__(self, model_dir='./models'):
        self.model_dir = model_dir
        self.model = None
        self.scaler = None
        self.feature_names = None
        self.config = None
        self.sequence_buffer = None
        
        self.load_model()
    
    def load_model(self):
        """Load the trained model and preprocessing components"""
        try:
            # Load configuration
            with open(f'{self.model_dir}/model_config.json', 'r') as f:
                self.config = json.load(f)
            
            # Load model
            model_path = f"{self.model_dir}/best_lstm_ids_{self.config['best_model']}.h5"
            self.model = keras.models.load_model(model_path)
            
            # Load scaler
            self.scaler = joblib.load(f'{self.model_dir}/feature_scaler.pkl')
            
            # Load feature names
            with open(f'{self.model_dir}/feature_names.txt', 'r') as f:
                self.feature_names = [line.strip() for line in f.readlines()]
            
            # Initialize sequence buffer
            self.sequence_buffer = deque(maxlen=self.config['sequence_length'])
            
            print(f"Model loaded successfully: {self.config['best_model']}")
            print(f"Features: {len(self.feature_names)}")
            print(f"Sequence length: {self.config['sequence_length']}")
            
        except Exception as e:
            print(f"Error loading model: {e}")
            raise
    
    def predict(self, features):
        """Make prediction on new features"""
        if self.model is None:
            raise ValueError("Model not loaded")
        
        # Ensure features is numpy array
        if isinstance(features, list):
            features = np.array(features)
        
        # Scale features
        features_scaled = self.scaler.transform(features.reshape(1, -1))[0]
        
        # Add to sequence buffer
        self.sequence_buffer.append(features_scaled)
        
        # Need full sequence for prediction
        if len(self.sequence_buffer) < self.config['sequence_length']:
            return {
                'prediction': 0,
                'probability': 0.0,
                'confidence': 0.0,
                'status': 'insufficient_data'
            }
        
        # Prepare sequence
        sequence = np.array(list(self.sequence_buffer)).reshape(1, self.config['sequence_length'], -1)
        
        # Make prediction
        probability = self.model.predict(sequence, verbose=0)[0][0]
        prediction = 1 if probability > self.config['threshold'] else 0
        confidence = abs(probability - self.config['threshold']) * 2
        
        return {
            'prediction': int(prediction),
            'probability': float(probability),
            'confidence': float(min(confidence, 1.0)),
            'status': 'success'
        }
    
    def get_feature_names(self):
        """Get list of expected feature names"""
        return self.feature_names.copy()
    
    def get_model_info(self):
        """Get model information"""
        return self.config.copy()
'''

with open('../models/lstm_model_loader.py', 'w') as f:
    f.write(deployment_script)

print("Created model loader: ../models/lstm_model_loader.py")
print("\nDeployment complete! You can now use the model in real-time applications.")

## Summary

This notebook demonstrated:

1. **Data Preprocessing**: Loading and preparing CIC-IDS dataset for LSTM training
2. **Feature Engineering**: Creating sequences and normalizing features
3. **Model Architecture**: Comparing different LSTM architectures
4. **Training**: Training models with proper validation and callbacks
5. **Evaluation**: Comprehensive performance analysis
6. **Real-time Testing**: Simulating real-time attack detection
7. **Deployment**: Creating deployment-ready model files

### Key Results:
- Best performing model: **{LSTM}**
- Real-time detection capability with sequence-based predictions
- Deployment-ready model files for integration

### Next Steps:
1. Integrate with `real_time_ids.py` for live packet capture
2. Deploy to production environment
3. Monitor and retrain model with new data
4. Implement automated response mechanisms